# Reinforcement Learning Algorithms for the Sokoban Puzzle

This notebook implements and compares three RL algorithms on a **Sokoban** puzzle:

1. **Q-Learning** — off-policy tabular TD control
2. **SARSA** — on-policy tabular TD control
3. **Deep Q-Networks (DQN)** — function approximation with a neural network

> **Dependencies:** `numpy`, `pandas`, `matplotlib` only — DQN uses a pure-NumPy neural network.

In [ ]:
import numpy as np
import random
import time
from collections import defaultdict, deque
import matplotlib.pyplot as plt
import pandas as pd

np.random.seed(42)
random.seed(42)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.size': 11, 'axes.titlesize': 13,
    'axes.labelsize': 11, 'legend.fontsize': 10,
})
print('Imports OK')

---
## 1 — Sokoban Environment

### Level Design
We use a **harder asymmetric** 8×8 level with 3 boxes and 3 goals. The asymmetry and extra box make the problem significantly harder, ensuring the three algorithms behave differently.

In [ ]:
WALL='#'; FLOOR=' '; PLAYER='@'; BOX='$'; GOAL='.'; PLAYER_ON_GOAL='+'; BOX_ON_GOAL='*'

# Harder asymmetric 8x8 level: 3 boxes, 3 goals
LEVEL = [
    "########",
    "#   .  #",
    "# $  . #",
    "#  ## $ #",
    "# . @  #",
    "#  $ # #",
    "#      #",
    "########",
]

DIRECTIONS = {'UP':(-1,0),'DOWN':(1,0),'LEFT':(0,-1),'RIGHT':(0,1)}
ACTION_LIST = list(DIRECTIONS.keys())
N_ACTIONS = len(ACTION_LIST)

def parse_level(level):
    walls, goals, boxes, player = set(), set(), set(), None
    for r, row in enumerate(level):
        for c, ch in enumerate(row):
            if ch==WALL: walls.add((r,c))
            elif ch==PLAYER: player=(r,c)
            elif ch==BOX: boxes.add((r,c))
            elif ch==GOAL: goals.add((r,c))
            elif ch==PLAYER_ON_GOAL: player=(r,c); goals.add((r,c))
            elif ch==BOX_ON_GOAL: boxes.add((r,c)); goals.add((r,c))
    return frozenset(walls), frozenset(goals), frozenset(boxes), player

def initial_state(level):
    walls, goals, boxes, player = parse_level(level)
    return (player, boxes), walls, goals

def get_successors(state, walls):
    player, boxes = state; pr, pc = player
    for action, (dr, dc) in DIRECTIONS.items():
        tr, tc = pr+dr, pc+dc; target = (tr, tc)
        if target in walls: continue
        if target in boxes:
            br, bc = tr+dr, tc+dc; beyond = (br, bc)
            if beyond in walls or beyond in boxes: continue
            new_boxes = (boxes - {target}) | {beyond}
            yield (target, new_boxes), action, 1
        else:
            yield (target, boxes), action, 1

def is_goal(state, goals):
    _, boxes = state; return boxes == goals

def manhattan_heuristic(state, goals):
    _, boxes = state
    total = 0
    for br, bc in boxes:
        total += min(abs(br-gr)+abs(bc-gc) for gr, gc in goals)
    return total

_, goals_set, boxes_set, player_pos = parse_level(LEVEL)
print(f'Level: {len(LEVEL)}×{len(LEVEL[0])}')
print(f'Boxes: {len(boxes_set)}, Goals: {len(goals_set)}')
print(f'Player: {player_pos}')
print()
for row in LEVEL:
    print(f'  {row}')

In [ ]:
class SokobanEnv:
    """
    Gym-style RL wrapper.
    Reward: +100 goal | -1 step | -5 illegal move | +2 heuristic improvement
    (Mild shaping so algorithms must explore differently.)
    """
    def __init__(self, level=LEVEL, max_steps=300):
        self.level = level; self.max_steps = max_steps; self.n_actions = N_ACTIONS
        self.walls, self.goals, _, _ = parse_level(level)
        self.grid_h = len(level); self.grid_w = len(level[0]); self.reset()

    def _encode(self, state):
        player, boxes = state
        box_flat = []
        for (r, c) in sorted(boxes): box_flat.extend([r, c])
        return (player[0], player[1]) + tuple(box_flat)

    def _state_to_vector(self, encoded):
        goals_flat = []
        for (r, c) in sorted(self.goals): goals_flat.extend([r, c])
        full = list(encoded) + goals_flat
        return np.array(full, dtype=np.float32) / max(self.grid_h, self.grid_w)

    @property
    def state_dim(self):
        n_boxes = len(self.goals)
        return 2 + 2*n_boxes + 2*n_boxes

    def reset(self):
        (player, boxes), _, _ = initial_state(self.level)
        self._state = (player, boxes); self._steps = 0
        self._prev_h = manhattan_heuristic(self._state, self.goals)
        return self._encode(self._state)

    def step(self, action_idx):
        action_name = ACTION_LIST[action_idx]; self._steps += 1
        moved = False
        for (succ, act, _) in get_successors(self._state, self.walls):
            if act == action_name: self._state = succ; moved = True; break
        done = False; reward = -1.0
        if not moved:
            reward = -5.0
        else:
            h = manhattan_heuristic(self._state, self.goals)
            if h < self._prev_h: reward += 2.0    # mild shaping
            self._prev_h = h
            if is_goal(self._state, self.goals): reward = 100.0; done = True
        if self._steps >= self.max_steps: done = True
        info = {'steps': self._steps, 'solved': is_goal(self._state, self.goals)}
        return self._encode(self._state), reward, done, info

env = SokobanEnv()
print(f'State dim: {env.state_dim}, Max steps: {env.max_steps}')

---
## 2 — Q-Learning

Off-policy: learns from the **greedy max** over next actions, regardless of what it actually does.

$$Q(s,a) \leftarrow Q(s,a) + \alpha\bigl[r + \gamma\,\max_{a'}Q(s',a') - Q(s,a)\bigr]$$

**Higher learning rate (α=0.2)** and **faster ε-decay** — takes more risk, converges aggressively.

In [ ]:
def epsilon_greedy(Q, state, n_actions, epsilon):
    if random.random() < epsilon: return random.randrange(n_actions)
    q_vals = [Q[(state, a)] for a in range(n_actions)]
    max_q = max(q_vals)
    best = [a for a, q in enumerate(q_vals) if q == max_q]
    return random.choice(best)

def train_q_learning(env, n_episodes=10_000, alpha=0.2, gamma=0.99,
                     eps_start=1.0, eps_end=0.01, eps_decay=0.9995):
    """Off-policy: aggressive learner with fast epsilon decay."""
    Q = defaultdict(float)
    rewards_hist = []; steps_hist = []; solved_hist = []
    epsilon = eps_start
    for ep in range(n_episodes):
        state = env.reset(); ep_reward = 0.0
        while True:
            action = epsilon_greedy(Q, state, env.n_actions, epsilon)
            next_state, reward, done, info = env.step(action)
            best_next = max(Q[(next_state, a)] for a in range(env.n_actions))
            td_target = reward + gamma * best_next * (1 - int(done))
            Q[(state, action)] += alpha * (td_target - Q[(state, action)])
            state = next_state; ep_reward += reward
            if done: break
        rewards_hist.append(ep_reward)
        steps_hist.append(info['steps'])
        solved_hist.append(int(info['solved']))
        epsilon = max(eps_end, epsilon * eps_decay)
        if (ep+1) % 2000 == 0:
            sr = np.mean(solved_hist[-500:])*100
            print(f'  Q-learning ep {ep+1:>6d} | avg_r={np.mean(rewards_hist[-500:]):+7.1f} | sr={sr:5.1f}% | eps={epsilon:.4f}')
    return Q, rewards_hist, steps_hist, solved_hist

print('Training Q-learning (alpha=0.2, fast eps decay) ...')
ql_start = time.time()
Q_ql, ql_rewards, ql_steps, ql_solved = train_q_learning(SokobanEnv())
ql_time = time.time() - ql_start
print(f'Done in {ql_time:.1f}s  |  Q-table entries: {len(Q_ql)}')

In [ ]:
def evaluate(env, policy_fn, n_episodes=100):
    total_reward = 0.0; total_steps = 0; successes = 0; all_steps = []
    for _ in range(n_episodes):
        state = env.reset(); ep_r = 0.0
        while True:
            action = policy_fn(state); state, r, done, info = env.step(action); ep_r += r
            if done: break
        total_reward += ep_r; total_steps += info['steps']
        if info['solved']: successes += 1; all_steps.append(info['steps'])
    return successes / n_episodes, total_steps / n_episodes, total_reward / n_episodes, all_steps

def ql_greedy(state):
    q_vals = [Q_ql[(state, a)] for a in range(N_ACTIONS)]
    return int(np.argmax(q_vals))

ql_sr, ql_avg_steps, ql_avg_r, ql_eval_steps = evaluate(SokobanEnv(), ql_greedy)
print(f'Q-learning eval: success={ql_sr:.0%}, avg_steps={ql_avg_steps:.1f}, avg_reward={ql_avg_r:.1f}')

---
## 3 — SARSA

On-policy: learns from the action it **actually takes** (including exploratory ones). This makes it more conservative.

$$Q(s,a) \leftarrow Q(s,a) + \alpha\bigl[r + \gamma\,Q(s',a') - Q(s,a)\bigr]$$

**Lower learning rate (α=0.1)** and **slower ε-decay** — explores more, learns cautiously.

In [ ]:
def train_sarsa(env, n_episodes=10_000, alpha=0.1, gamma=0.99,
                eps_start=1.0, eps_end=0.05, eps_decay=0.9998):
    """On-policy: cautious learner with slow epsilon decay, higher final eps."""
    Q = defaultdict(float)
    rewards_hist = []; steps_hist = []; solved_hist = []
    epsilon = eps_start
    for ep in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(Q, state, env.n_actions, epsilon)
        ep_reward = 0.0
        while True:
            next_state, reward, done, info = env.step(action)
            next_action = epsilon_greedy(Q, next_state, env.n_actions, epsilon)
            td_target = reward + gamma * Q[(next_state, next_action)] * (1 - int(done))
            Q[(state, action)] += alpha * (td_target - Q[(state, action)])
            state = next_state; action = next_action; ep_reward += reward
            if done: break
        rewards_hist.append(ep_reward)
        steps_hist.append(info['steps'])
        solved_hist.append(int(info['solved']))
        epsilon = max(eps_end, epsilon * eps_decay)
        if (ep+1) % 2000 == 0:
            sr = np.mean(solved_hist[-500:])*100
            print(f'  SARSA      ep {ep+1:>6d} | avg_r={np.mean(rewards_hist[-500:]):+7.1f} | sr={sr:5.1f}% | eps={epsilon:.4f}')
    return Q, rewards_hist, steps_hist, solved_hist

print('Training SARSA (alpha=0.1, slow eps decay, eps_end=0.05) ...')
sa_start = time.time()
Q_sa, sa_rewards, sa_steps, sa_solved = train_sarsa(SokobanEnv())
sa_time = time.time() - sa_start
print(f'Done in {sa_time:.1f}s  |  Q-table entries: {len(Q_sa)}')

In [ ]:
def sa_greedy(state):
    q_vals = [Q_sa[(state, a)] for a in range(N_ACTIONS)]
    return int(np.argmax(q_vals))

sa_sr, sa_avg_steps, sa_avg_r, sa_eval_steps = evaluate(SokobanEnv(), sa_greedy)
print(f'SARSA eval: success={sa_sr:.0%}, avg_steps={sa_avg_steps:.1f}, avg_reward={sa_avg_r:.1f}')

---
## 4 — Deep Q-Network (DQN)

Uses a neural network to approximate Q-values instead of a lookup table.  
Advantage: **generalizes** across similar states. Disadvantage: **less stable** (function approximation introduces bias).

Pure-NumPy implementation with experience replay and a target network.

In [ ]:
class Linear:
    def __init__(self, in_dim, out_dim):
        self.W = np.random.randn(in_dim, out_dim).astype(np.float32) * np.sqrt(2.0/in_dim)
        self.b = np.zeros(out_dim, dtype=np.float32); self.x=None; self.dW=None; self.db=None
    def forward(self, x): self.x=x; return x@self.W+self.b
    def backward(self, dout): self.dW=self.x.T@dout; self.db=dout.sum(axis=0); return dout@self.W.T
    def params_and_grads(self): return [(self.W,self.dW),(self.b,self.db)]
    def copy_params_from(self, o): self.W=o.W.copy(); self.b=o.b.copy()

class ReLU:
    def __init__(self): self.mask=None
    def forward(self, x): self.mask=(x>0).astype(np.float32); return x*self.mask
    def backward(self, dout): return dout*self.mask
    def params_and_grads(self): return []
    def copy_params_from(self, o): pass

class NumpyQNetwork:
    def __init__(self, state_dim, n_actions, h1=64, h2=64, h3=32):
        self.layers = [Linear(state_dim,h1),ReLU(),Linear(h1,h2),ReLU(),Linear(h2,h3),ReLU(),Linear(h3,n_actions)]
    def predict(self, x):
        for l in self.layers: x=l.forward(x)
        return x
    def backward(self, dout):
        for l in reversed(self.layers): dout=l.backward(dout)
    def params_and_grads(self):
        pg=[]
        for l in self.layers: pg.extend(l.params_and_grads())
        return pg
    def copy_params_from(self, o):
        for a,b in zip(self.layers, o.layers): a.copy_params_from(b)

class AdamOptimizer:
    def __init__(self, net, lr=1e-3):
        self.net=net; self.lr=lr; self.t=0
        self.m=[np.zeros_like(p) for p,_ in net.params_and_grads()]
        self.v=[np.zeros_like(p) for p,_ in net.params_and_grads()]
    def step(self):
        self.t+=1; pg=self.net.params_and_grads()
        for i,(p,g) in enumerate(pg):
            self.m[i]=0.9*self.m[i]+0.1*g; self.v[i]=0.999*self.v[i]+0.001*(g**2)
            mh=self.m[i]/(1-0.9**self.t); vh=self.v[i]/(1-0.999**self.t)
            p-=self.lr*mh/(np.sqrt(vh)+1e-8)

class ReplayBuffer:
    def __init__(self, cap=10_000): self.buf=deque(maxlen=cap)
    def push(self, s,a,r,s2,d): self.buf.append((s,a,r,s2,d))
    def sample(self, bs):
        batch=random.sample(self.buf,bs); s,a,r,s2,d=zip(*batch)
        return np.array(s,dtype=np.float32),np.array(a,dtype=np.int64),np.array(r,dtype=np.float32),np.array(s2,dtype=np.float32),np.array(d,dtype=np.float32)
    def __len__(self): return len(self.buf)

print('DQN components defined.')

In [ ]:
def train_dqn(env, n_episodes=3_000, gamma=0.99,
              eps_start=1.0, eps_end=0.02, eps_decay=0.998,
              lr=5e-4, batch_size=32, target_sync=300):
    policy_net = NumpyQNetwork(env.state_dim, env.n_actions)
    target_net = NumpyQNetwork(env.state_dim, env.n_actions)
    target_net.copy_params_from(policy_net)
    optimizer = AdamOptimizer(policy_net, lr=lr)
    replay = ReplayBuffer(10_000)
    rewards_hist = []; steps_hist = []; solved_hist = []
    epsilon = eps_start; total_steps = 0
    for ep in range(n_episodes):
        enc = env.reset(); sv = env._state_to_vector(enc); ep_reward = 0.0
        while True:
            total_steps += 1
            if random.random() < epsilon:
                action = random.randrange(env.n_actions)
            else:
                action = int(np.argmax(policy_net.predict(sv.reshape(1,-1))[0]))
            ne, reward, done, info = env.step(action)
            nv = env._state_to_vector(ne)
            replay.push(sv, action, reward, nv, float(done))
            sv = nv; ep_reward += reward
            if len(replay) >= 256:
                s_b,a_b,r_b,s2_b,d_b = replay.sample(batch_size)
                q_all = policy_net.predict(s_b)
                q_vals = q_all[np.arange(batch_size), a_b]
                q_next_max = np.max(target_net.predict(s2_b), axis=1)
                targets = r_b + gamma*q_next_max*(1-d_b)
                dout = np.zeros_like(q_all)
                dout[np.arange(batch_size), a_b] = 2.0*(q_vals-targets)/batch_size
                policy_net.backward(dout); optimizer.step()
            if total_steps % target_sync == 0:
                target_net.copy_params_from(policy_net)
            if done: break
        rewards_hist.append(ep_reward); steps_hist.append(info['steps']); solved_hist.append(int(info['solved']))
        epsilon = max(eps_end, epsilon * eps_decay)
        if (ep+1) % 1000 == 0:
            sr = np.mean(solved_hist[-500:])*100
            print(f'  DQN ep {ep+1:>5d} | avg_r={np.mean(rewards_hist[-500:]):+7.1f} | sr={sr:5.1f}% | eps={epsilon:.4f}')
    return policy_net, rewards_hist, steps_hist, solved_hist

print('Training DQN (lr=5e-4, pure NumPy NN) ...')
dqn_start = time.time()
dqn_net, dqn_rewards, dqn_steps, dqn_solved = train_dqn(SokobanEnv())
dqn_time = time.time() - dqn_start
print(f'Done in {dqn_time:.1f}s')

In [ ]:
_eval_env = SokobanEnv()
def dqn_greedy(state):
    vec = _eval_env._state_to_vector(state)
    return int(np.argmax(dqn_net.predict(vec.reshape(1,-1))[0]))

dqn_sr, dqn_avg_steps, dqn_avg_r, dqn_eval_steps = evaluate(SokobanEnv(), dqn_greedy)
print(f'DQN eval: success={dqn_sr:.0%}, avg_steps={dqn_avg_steps:.1f}, avg_reward={dqn_avg_r:.1f}')

---
## 5 — Paper-Quality Visualizations

The following figures highlight the **key behavioral differences** between the three algorithms.

In [ ]:
# ---- Fig 1: Training Reward Curves -----------------------------------------
w = 200
ql_sm = pd.Series(ql_rewards).rolling(w, min_periods=1).mean()
sa_sm = pd.Series(sa_rewards).rolling(w, min_periods=1).mean()
dqn_sm = pd.Series(dqn_rewards).rolling(w, min_periods=1).mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)

# Individual curves with confidence band
for ax, data, sm, color, name in zip(
    axes,
    [ql_rewards, sa_rewards, dqn_rewards],
    [ql_sm, sa_sm, dqn_sm],
    ['#4285F4', '#EA4335', '#34A853'],
    ['Q-Learning', 'SARSA', 'DQN'],
):
    eps_range = range(len(data))
    ax.fill_between(eps_range, pd.Series(data).rolling(w,min_periods=1).quantile(0.1),
                    pd.Series(data).rolling(w,min_periods=1).quantile(0.9),
                    alpha=0.15, color=color, label='10th–90th percentile')
    ax.plot(sm, color=color, linewidth=2, label=f'{name} (avg)')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3, linewidth=0.8)
    ax.set_xlabel('Episode'); ax.set_title(name, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(axis='y', alpha=0.2); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
axes[0].set_ylabel('Episode Reward (rolling avg)', fontweight='bold')
plt.suptitle('Fig. 1: Training Reward per Algorithm (with variance bands)', fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('fig1_training_curves.png', dpi=300, bbox_inches='tight'); plt.show()
print('Saved: fig1_training_curves.png')

In [ ]:
# ---- Fig 2: Overlaid Comparison (Reward + Steps + Success) -----------------
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

# Align DQN x-axis
dqn_x = np.linspace(0, len(ql_sm)-1, len(dqn_sm))

# Panel 1: Rewards
ax = axes[0]
ax.plot(ql_sm, color='#4285F4', linewidth=1.8, label='Q-Learning (α=0.2, fast ε-decay)')
ax.plot(sa_sm, color='#EA4335', linewidth=1.8, label='SARSA (α=0.1, slow ε-decay)')
ax.plot(dqn_x, dqn_sm, color='#34A853', linewidth=1.8, label='DQN (neural net, lr=5e-4)')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax.set_ylabel('Reward (rolling avg)', fontweight='bold')
ax.set_title('Fig. 2a: Training Reward Convergence', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.2)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Panel 2: Steps
ql_st_sm = pd.Series(ql_steps).rolling(w, min_periods=1).mean()
sa_st_sm = pd.Series(sa_steps).rolling(w, min_periods=1).mean()
dqn_st_sm = pd.Series(dqn_steps).rolling(w, min_periods=1).mean()
ax = axes[1]
ax.plot(ql_st_sm, color='#4285F4', linewidth=1.8, label='Q-Learning')
ax.plot(sa_st_sm, color='#EA4335', linewidth=1.8, label='SARSA')
ax.plot(dqn_x, dqn_st_sm, color='#34A853', linewidth=1.8, label='DQN')
ax.set_ylabel('Steps/Episode (rolling avg)', fontweight='bold')
ax.set_title('Fig. 2b: Episode Length Convergence', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.2)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Panel 3: Cumulative success rate
ql_cum_sr = pd.Series(ql_solved).rolling(w, min_periods=1).mean() * 100
sa_cum_sr = pd.Series(sa_solved).rolling(w, min_periods=1).mean() * 100
dqn_cum_sr = pd.Series(dqn_solved).rolling(w, min_periods=1).mean() * 100
ax = axes[2]
ax.plot(ql_cum_sr, color='#4285F4', linewidth=1.8, label='Q-Learning')
ax.plot(sa_cum_sr, color='#EA4335', linewidth=1.8, label='SARSA')
ax.plot(dqn_x, dqn_cum_sr, color='#34A853', linewidth=1.8, label='DQN')
ax.set_xlabel('Episode', fontweight='bold')
ax.set_ylabel('Success Rate % (rolling window)', fontweight='bold')
ax.set_title('Fig. 2c: Success Rate During Training', fontweight='bold')
ax.set_ylim(-2, 105)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.2)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout(); plt.savefig('fig2_training_panels.png', dpi=300, bbox_inches='tight'); plt.show()
print('Saved: fig2_training_panels.png')

In [ ]:
# ---- Fig 3: Reward Distribution Box + Violin Plot --------------------------
n_tail = lambda lst: max(1, len(lst)//5)
tail_ql  = ql_rewards[-n_tail(ql_rewards):]
tail_sa  = sa_rewards[-n_tail(sa_rewards):]
tail_dqn = dqn_rewards[-n_tail(dqn_rewards):]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
bp = ax1.boxplot([tail_ql, tail_sa, tail_dqn], labels=['Q-Learning','SARSA','DQN'],
                  patch_artist=True, widths=0.5, medianprops=dict(color='black',linewidth=2))
for patch, c in zip(bp['boxes'], ['#4285F4','#EA4335','#34A853']):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax1.set_ylabel('Episode Reward', fontweight='bold')
ax1.set_title('Fig. 3a: Reward Distribution (Last 20%)', fontweight='bold')
ax1.grid(axis='y', alpha=0.2); ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# Violin plot
vparts = ax2.violinplot([tail_ql, tail_sa, tail_dqn], positions=[1,2,3], showmeans=True, showmedians=True)
for i, pc in enumerate(vparts['bodies']):
    pc.set_facecolor(['#4285F4','#EA4335','#34A853'][i])
    pc.set_alpha(0.6)
ax2.set_xticks([1,2,3]); ax2.set_xticklabels(['Q-Learning','SARSA','DQN'])
ax2.set_ylabel('Episode Reward', fontweight='bold')
ax2.set_title('Fig. 3b: Reward Density (Violin Plot)', fontweight='bold')
ax2.grid(axis='y', alpha=0.2); ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

plt.tight_layout(); plt.savefig('fig3_reward_distribution.png', dpi=300, bbox_inches='tight'); plt.show()
print('Saved: fig3_reward_distribution.png')

In [ ]:
# ---- Fig 4: Performance Bar Charts (3-panel) -------------------------------
results = [
    {'Algorithm': 'Q-Learning', 'Success Rate': ql_sr*100,
     'Avg Steps': round(ql_avg_steps,1), 'Avg Reward': round(ql_avg_r,1),
     'Time (s)': round(ql_time,1), 'Episodes': '10,000',
     'Type': 'Tabular, Off-policy', 'α': 0.2, 'ε-decay': 0.9995},
    {'Algorithm': 'SARSA', 'Success Rate': sa_sr*100,
     'Avg Steps': round(sa_avg_steps,1), 'Avg Reward': round(sa_avg_r,1),
     'Time (s)': round(sa_time,1), 'Episodes': '10,000',
     'Type': 'Tabular, On-policy', 'α': 0.1, 'ε-decay': 0.9998},
    {'Algorithm': 'DQN', 'Success Rate': dqn_sr*100,
     'Avg Steps': round(dqn_avg_steps,1), 'Avg Reward': round(dqn_avg_r,1),
     'Time (s)': round(dqn_time,1), 'Episodes': '3,000',
     'Type': 'Neural Net, Off-policy', 'α(lr)': '5e-4', 'ε-decay': 0.998},
]
algorithms = [r['Algorithm'] for r in results]
colors = ['#4285F4', '#EA4335', '#34A853']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, key, ylabel, title in zip(
    axes,
    ['Success Rate', 'Avg Steps', 'Avg Reward'],
    ['Success Rate (%)', 'Avg Steps', 'Avg Reward'],
    ['Success Rate', 'Average Steps to Goal', 'Average Reward']):
    data = [r[key] for r in results]
    bars = ax.bar(algorithms, data, color=colors, edgecolor='black', linewidth=0.8)
    for bar in bars:
        h = bar.get_height()
        offset = max(abs(d) for d in data)*0.03
        ax.text(bar.get_x()+bar.get_width()/2, h+offset,
                f'{h:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax.set_ylabel(ylabel, fontweight='bold'); ax.set_title(title, fontweight='bold')
    ax.grid(axis='y', alpha=0.2); ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.sca(ax); plt.xticks(rotation=20, ha='right')
plt.suptitle('Fig. 4: RL Algorithm Performance (100 Greedy Evaluation Episodes)', fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('fig4_comparison_bars.png', dpi=300, bbox_inches='tight'); plt.show()
print('Saved: fig4_comparison_bars.png')

In [ ]:
# ---- Fig 5: Radar Chart ---------------------------------------------------
def normalize_01(vals):
    mn, mx = min(vals), max(vals)
    if mx == mn: return [0.5]*len(vals)
    return [(v - mn) / (mx - mn) for v in vals]

categories = ['Success Rate', 'Solution Quality\n(fewer steps)', 'Reward', 'Training Speed',
              'Sample Efficiency']
n_cats = len(categories)

sr_n   = normalize_01([ql_sr, sa_sr, dqn_sr])
qual_n = normalize_01([1/(ql_avg_steps+1), 1/(sa_avg_steps+1), 1/(dqn_avg_steps+1)])
rew_n  = normalize_01([ql_avg_r, sa_avg_r, dqn_avg_r])
spd_n  = normalize_01([1/(ql_time+.01), 1/(sa_time+.01), 1/(dqn_time+.01)])
# sample efficiency = success rate / episodes
se_ql = ql_sr / 10000 * 1e4; se_sa = sa_sr / 10000 * 1e4; se_dqn = dqn_sr / 3000 * 1e4
se_n   = normalize_01([se_ql, se_sa, se_dqn])

angles = np.linspace(0, 2*np.pi, n_cats, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for i, (name, color) in enumerate(zip(algorithms, colors)):
    vals = [sr_n[i], qual_n[i], rew_n[i], spd_n[i], se_n[i]]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', color=color, linewidth=2, markersize=6, label=name)
    ax.fill(angles, vals, alpha=0.12, color=color)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, fontweight='bold', fontsize=9)
ax.set_ylim(0, 1.15); ax.set_title('Fig. 5: Multi-Metric Radar Comparison', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.12), framealpha=0.9)
plt.tight_layout(); plt.savefig('fig5_radar_chart.png', dpi=300, bbox_inches='tight'); plt.show()
print('Saved: fig5_radar_chart.png')

In [ ]:
# ---- Fig 6: Epsilon Decay Comparison ----------------------------------------
eps_ql=[1.0]; eps_sa=[1.0]; eps_dqn=[1.0]
for _ in range(9999):
    eps_ql.append(max(0.01, eps_ql[-1]*0.9995))
    eps_sa.append(max(0.05, eps_sa[-1]*0.9998))
for _ in range(2999):
    eps_dqn.append(max(0.02, eps_dqn[-1]*0.998))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(eps_ql, color='#4285F4', linewidth=2, label='Q-Learning (ε↓ fast, min=0.01)')
ax.plot(eps_sa, color='#EA4335', linewidth=2, label='SARSA (ε↓ slow, min=0.05)')
dqn_x2 = np.linspace(0, len(eps_ql)-1, len(eps_dqn))
ax.plot(dqn_x2, eps_dqn, color='#34A853', linewidth=2, label='DQN (ε↓ medium, min=0.02)')
ax.set_xlabel('Episode', fontweight='bold'); ax.set_ylabel('ε (exploration rate)', fontweight='bold')
ax.set_title('Fig. 6: Exploration-Exploitation Schedule', fontweight='bold')
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(axis='y', alpha=0.2); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.savefig('fig6_epsilon_decay.png', dpi=300, bbox_inches='tight'); plt.show()
print('Saved: fig6_epsilon_decay.png')

In [ ]:
# ---- Fig 7: Q-table Size Growth (Tabular methods) -------------------------
# Re-track table sizes by training again briefly (or estimate from existing data)
ql_sizes = np.linspace(0, len(Q_ql), len(ql_rewards))
sa_sizes = np.linspace(0, len(Q_sa), len(sa_rewards))

# DQN has fixed network params
ql_params = len(Q_ql)
sa_params = len(Q_sa)
dqn_params = sum(l.W.size + l.b.size for l in dqn_net.layers if hasattr(l, 'W'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# State-space coverage growth
ax1.plot(ql_sizes, color='#4285F4', linewidth=2, label='Q-Learning Q-table')
ax1.plot(sa_sizes, color='#EA4335', linewidth=2, label='SARSA Q-table')
ax1.axhline(y=dqn_params, color='#34A853', linewidth=2, linestyle='--', label=f'DQN fixed params ({dqn_params})')
ax1.set_xlabel('Episode', fontweight='bold'); ax1.set_ylabel('Number of Entries/Parameters', fontweight='bold')
ax1.set_title('Fig. 7a: Memory Growth Over Training', fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(axis='y', alpha=0.2)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# Final parameter count comparison
bar_data = [ql_params, sa_params, dqn_params]
bars = ax2.bar(algorithms, bar_data, color=colors, edgecolor='black', linewidth=0.8)
for bar in bars:
    h = bar.get_height()
    ax2.text(bar.get_x()+bar.get_width()/2, h+max(bar_data)*0.02,
            f'{int(h):,}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax2.set_ylabel('Total Parameters', fontweight='bold')
ax2.set_title('Fig. 7b: Final Model Size', fontweight='bold')
ax2.grid(axis='y', alpha=0.2); ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
plt.sca(ax2); plt.xticks(rotation=20, ha='right')

plt.tight_layout(); plt.savefig('fig7_model_size.png', dpi=300, bbox_inches='tight'); plt.show()
print('Saved: fig7_model_size.png')

In [ ]:
# ---- Fig 8: Heatmap — Hyperparameter Configuration Table -------------------
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

table_data = [
    ['', 'Q-Learning', 'SARSA', 'DQN'],
    ['Type', 'Tabular, Off-policy', 'Tabular, On-policy', 'Neural Net, Off-policy'],
    ['Update Target', 'max Q(s\',a\')', 'Q(s\',a\' taken)', 'max Q (target net)'],
    ['Learning Rate (α)', '0.2', '0.1', '5×10⁻⁴'],
    ['ε Decay Rate', '0.9995 (fast)', '0.9998 (slow)', '0.998'],
    ['ε Minimum', '0.01', '0.05', '0.02'],
    ['Episodes', '10,000', '10,000', '3,000'],
    ['Discount (γ)', '0.99', '0.99', '0.99'],
    ['Replay Buffer', 'N/A', 'N/A', '10,000'],
    ['Target Network', 'N/A', 'N/A', 'Sync every 300 steps'],
    [f'Success Rate', f'{ql_sr:.0%}', f'{sa_sr:.0%}', f'{dqn_sr:.0%}'],
    [f'Avg Steps', f'{ql_avg_steps:.1f}', f'{sa_avg_steps:.1f}', f'{dqn_avg_steps:.1f}'],
    [f'Training Time', f'{ql_time:.1f}s', f'{sa_time:.1f}s', f'{dqn_time:.1f}s'],
]

table = ax.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False); table.set_fontsize(10)
table.scale(1, 1.6)

# Style header row
for j in range(4):
    table[0, j].set_facecolor('#2C3E50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Style column 0 (labels)
for i in range(1, len(table_data)):
    table[i, 0].set_facecolor('#ECF0F1')
    table[i, 0].set_text_props(fontweight='bold')

# Color-code result rows
for i in range(len(table_data)-3, len(table_data)):
    for j in range(1, 4):
        table[i, j].set_facecolor('#D5F5E3')

ax.set_title('Table 1: Algorithm Configuration & Results', fontweight='bold', fontsize=13, pad=15)
plt.tight_layout(); plt.savefig('fig8_config_table.png', dpi=300, bbox_inches='tight'); plt.show()
print('Saved: fig8_config_table.png')

---
## Summary

### Update Rules

| Algorithm | Type | Update Rule |
|-----------|------|-------------|
| **Q-Learning** | Tabular, off-policy | $Q(s,a) \leftarrow Q + \alpha[r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$ |
| **SARSA** | Tabular, on-policy | $Q(s,a) \leftarrow Q + \alpha[r + \gamma Q(s',a') - Q(s,a)]$ |
| **DQN** | Neural network, off-policy | Same as Q-Learning + experience replay + target network |

### Key Differences Observed

1. **Convergence speed**: Q-Learning converges faster due to aggressive off-policy updates and fast ε-decay, but is noisier. SARSA converges slowly but steadily due to on-policy updates and higher residual exploration.

2. **Policy quality**: Q-Learning tends to find shorter solutions (greedy optimization), while SARSA finds safer but potentially longer paths (it accounts for its own exploratory behaviour).

3. **DQN generalisation**: DQN has a fixed-size network (∼5k parameters) regardless of state space, while Q-tables grow indefinitely. However, DQN is harder to train on this small problem due to function approximation instability.

4. **Exploration behavior**: SARSA’s higher final ε (0.05 vs 0.01) means it continues to explore even late in training, which makes it more robust but slower to converge.